# A3-A6

In [1]:
import gmsh

# -------------------------------------------------------------------------
# Geometry dimensions [m]
# -------------------------------------------------------------------------

Lx = 500.0
Ly = 750.0
Lz = 125.0

# -------------------------------------------------------------------------
# Crevasse dimensions [m]
# -------------------------------------------------------------------------

lx = 5.0
ly = 10.0
lz = 10.0

# -------------------------------------------------------------------------
# Crevasse offsets
# -------------------------------------------------------------------------

x_offsets = [15.0, 25.0, 50.0, 100.0]

# -------------------------------------------------------------------------
# Mesh size controls [m]
# -------------------------------------------------------------------------

h_min = 1.0
h_max = 20

# Near crack-tip refinement
h_refined = 2.5

refinement_x = 10.0
refinement_y = 10.0
refinement_z = 10.0

transition_thickness = 10.0

# -------------------------------------------------------------------------
# File numbering
# -------------------------------------------------------------------------

file_index = {
    15.0: 3,
    25.0: 4,
    50.0: 5,
    100.0: 6,
}

# -------------------------------------------------------------------------
# Generate meshes
# -------------------------------------------------------------------------

for x_offset in x_offsets:

    gmsh.initialize()

    gmsh.model.add(
        "glacier_terminus_{:.0f}".format(x_offset)
    )

    # ---------------------------------------------------------------------
    # Global mesh settings
    # ---------------------------------------------------------------------

    gmsh.option.setNumber(
        "Mesh.MeshSizeMin",
        h_min,
    )

    gmsh.option.setNumber(
        "Mesh.MeshSizeMax",
        h_max,
    )

    gmsh.option.setNumber(
        "Mesh.MeshSizeFactor",
        1.0,
    )

    # ---------------------------------------------------------------------
    # Glacier geometry
    # ---------------------------------------------------------------------

    glacier = gmsh.model.occ.addBox(
        0.0,
        0.0,
        0.0,
        Lx,
        Ly,
        Lz,
    )

    # ---------------------------------------------------------------------
    # Crevasse 1
    #
    # Enters from y = 0
    # ---------------------------------------------------------------------

    crevasse_1_x0 = (
        0.5 * Lx
        - x_offset
        - 0.5 * lx
    )

    crevasse_1_y0 = 0.0
    crevasse_1_z0 = Lz - lz

    crevasse_1 = gmsh.model.occ.addBox(
        crevasse_1_x0,
        crevasse_1_y0,
        crevasse_1_z0,
        lx,
        ly,
        lz,
    )

    # ---------------------------------------------------------------------
    # Crevasse 2
    #
    # Enters from y = Ly
    # ---------------------------------------------------------------------

    crevasse_2_x0 = (
        0.5 * Lx
        + x_offset
        - 0.5 * lx
    )

    crevasse_2_y0 = Ly - ly
    crevasse_2_z0 = Lz - lz

    crevasse_2 = gmsh.model.occ.addBox(
        crevasse_2_x0,
        crevasse_2_y0,
        crevasse_2_z0,
        lx,
        ly,
        lz,
    )

    # ---------------------------------------------------------------------
    # Subtract both crevasses
    # ---------------------------------------------------------------------

    domain, _ = gmsh.model.occ.cut(
        [(3, glacier)],
        [
            (3, crevasse_1),
            (3, crevasse_2),
        ],
        removeObject=True,
        removeTool=True,
    )

    gmsh.model.occ.synchronize()

    # ---------------------------------------------------------------------
    # Internal planes
    #
    # z = Lz / 4 = 31.25 m
    # z = Lz / 2 = 62.50 m
    # ---------------------------------------------------------------------

    z_quarter = Lz / 4.0
    z_half = Lz / 2.0

    plane_quarter = gmsh.model.occ.addRectangle(
        0.0,
        0.0,
        z_quarter,
        Lx,
        Ly,
    )

    plane_half = gmsh.model.occ.addRectangle(
        0.0,
        0.0,
        z_half,
        Lx,
        Ly,
    )

    gmsh.model.occ.synchronize()

    # ---------------------------------------------------------------------
    # Fragment glacier using internal planes
    # ---------------------------------------------------------------------

    domain, _ = gmsh.model.occ.fragment(
        domain,
        [
            (2, plane_quarter),
            (2, plane_half),
        ],
    )

    gmsh.model.occ.synchronize()

    # ---------------------------------------------------------------------
    # Physical volume
    # ---------------------------------------------------------------------

    volume_tags = [
        tag
        for dim, tag in domain
        if dim == 3
    ]

    gmsh.model.addPhysicalGroup(
        3,
        volume_tags,
        1,
    )

    gmsh.model.setPhysicalName(
        3,
        1,
        "GLACIER",
    )

    # ---------------------------------------------------------------------
    # Crack-tip coordinates
    #
    # Crevasse 1 tip:
    #     y = ly
    #
    # Crevasse 2 tip:
    #     y = Ly - ly
    #
    # z tip corresponds to bottom of notch
    # ---------------------------------------------------------------------

    crack_1_tip_x = 0.5 * Lx - x_offset
    crack_1_tip_y = ly
    crack_1_tip_z = Lz - lz

    crack_2_tip_x = 0.5 * Lx + x_offset
    crack_2_tip_y = Ly - ly
    crack_2_tip_z = Lz - lz

    # ---------------------------------------------------------------------
    # Refinement box around crack tip 1
    # ---------------------------------------------------------------------

    field_box_1 = gmsh.model.mesh.field.add(
        "Box"
    )

    gmsh.model.mesh.field.setNumber(
        field_box_1,
        "VIn",
        h_refined,
    )

    gmsh.model.mesh.field.setNumber(
        field_box_1,
        "VOut",
        h_max,
    )

    gmsh.model.mesh.field.setNumber(
        field_box_1,
        "XMin",
        crack_1_tip_x - refinement_x,
    )

    gmsh.model.mesh.field.setNumber(
        field_box_1,
        "XMax",
        crack_1_tip_x + refinement_x,
    )

    gmsh.model.mesh.field.setNumber(
        field_box_1,
        "YMin",
        max(
            0.0,
            crack_1_tip_y - refinement_y,
        ),
    )

    gmsh.model.mesh.field.setNumber(
        field_box_1,
        "YMax",
        min(
            Ly,
            crack_1_tip_y + refinement_y,
        ),
    )

    gmsh.model.mesh.field.setNumber(
        field_box_1,
        "ZMin",
        max(
            0.0,
            crack_1_tip_z - refinement_z,
        ),
    )

    gmsh.model.mesh.field.setNumber(
        field_box_1,
        "ZMax",
        Lz,
    )

    gmsh.model.mesh.field.setNumber(
        field_box_1,
        "Thickness",
        transition_thickness,
    )

    # ---------------------------------------------------------------------
    # Refinement box around crack tip 2
    # ---------------------------------------------------------------------

    field_box_2 = gmsh.model.mesh.field.add(
        "Box"
    )

    gmsh.model.mesh.field.setNumber(
        field_box_2,
        "VIn",
        h_refined,
    )

    gmsh.model.mesh.field.setNumber(
        field_box_2,
        "VOut",
        h_max,
    )

    gmsh.model.mesh.field.setNumber(
        field_box_2,
        "XMin",
        crack_2_tip_x - refinement_x,
    )

    gmsh.model.mesh.field.setNumber(
        field_box_2,
        "XMax",
        crack_2_tip_x + refinement_x,
    )

    gmsh.model.mesh.field.setNumber(
        field_box_2,
        "YMin",
        max(
            0.0,
            crack_2_tip_y - refinement_y,
        ),
    )

    gmsh.model.mesh.field.setNumber(
        field_box_2,
        "YMax",
        min(
            Ly,
            crack_2_tip_y + refinement_y,
        ),
    )

    gmsh.model.mesh.field.setNumber(
        field_box_2,
        "ZMin",
        max(
            0.0,
            crack_2_tip_z - refinement_z,
        ),
    )

    gmsh.model.mesh.field.setNumber(
        field_box_2,
        "ZMax",
        Lz,
    )

    gmsh.model.mesh.field.setNumber(
        field_box_2,
        "Thickness",
        transition_thickness,
    )

    # ---------------------------------------------------------------------
    # Combine both refinement fields
    # ---------------------------------------------------------------------

    field_min = gmsh.model.mesh.field.add(
        "Min"
    )

    gmsh.model.mesh.field.setNumbers(
        field_min,
        "FieldsList",
        [
            field_box_1,
            field_box_2,
        ],
    )

    gmsh.model.mesh.field.setAsBackgroundMesh(
        field_min
    )

    # ---------------------------------------------------------------------
    # Mesh settings
    # ---------------------------------------------------------------------

    gmsh.option.setNumber(
        "Mesh.Algorithm3D",
        10,
    )

    gmsh.option.setNumber(
        "Mesh.MeshSizeFromCurvature",
        0,
    )

    gmsh.option.setNumber(
        "Mesh.MeshSizeFromPoints",
        0,
    )

    gmsh.option.setNumber(
        "Mesh.MeshSizeExtendFromBoundary",
        0,
    )

    # ---------------------------------------------------------------------
    # Generate tetrahedral mesh
    # ---------------------------------------------------------------------

    gmsh.model.mesh.generate(3)

    # ---------------------------------------------------------------------
    # Output filename
    #
    # 03_Lx500_2C_S15.msh
    # 04_Lx500_2C_S25.msh
    # 05_Lx500_2C_S50.msh
    # 06_Lx500_2C_S100.msh
    # ---------------------------------------------------------------------

    index = file_index[x_offset]

    filename = (
        f"{index:02d}/"
        f"Lx{Lx:.0f}_"
        f"2C_"
        f"S{x_offset:.0f}.msh"
    )

    gmsh.write(filename)

    print(f"Generated: {filename}")

    gmsh.finalize()

Info    : Meshing 1D...nts - Looking for internal shapes                                                                                
Info    : [  0%] Meshing curve 50 (Line)
Info    : [ 10%] Meshing curve 51 (Line)
Info    : [ 10%] Meshing curve 52 (Line)
Info    : [ 10%] Meshing curve 53 (Line)
Info    : [ 10%] Meshing curve 54 (Line)
Info    : [ 10%] Meshing curve 55 (Line)
Info    : [ 20%] Meshing curve 56 (Line)
Info    : [ 20%] Meshing curve 57 (Line)
Info    : [ 20%] Meshing curve 58 (Line)
Info    : [ 20%] Meshing curve 59 (Line)
Info    : [ 20%] Meshing curve 60 (Line)
Info    : [ 30%] Meshing curve 61 (Line)
Info    : [ 30%] Meshing curve 62 (Line)
Info    : [ 30%] Meshing curve 63 (Line)
Info    : [ 30%] Meshing curve 64 (Line)
Info    : [ 30%] Meshing curve 65 (Line)
Info    : [ 40%] Meshing curve 66 (Line)
Info    : [ 40%] Meshing curve 67 (Line)
Info    : [ 40%] Meshing curve 68 (Line)
Info    : [ 40%] Meshing curve 69 (Line)
Info    : [ 40%] Meshing curve 70 (Line)
In

# XDMF - A3-A6

In [2]:
import meshio

mesh_files = ["03/Lx500_2C_S15.msh", "04/Lx500_2C_S25.msh", "05/Lx500_2C_S50.msh", "06/Lx500_2C_S100.msh"]

for mesh_file in mesh_files:
    mesh = meshio.read(mesh_file)
    cells = mesh.get_cells_type("tetra")
    points = mesh.points

    meshio.write(
        mesh_file.replace(".msh", ".xdmf"),
        meshio.Mesh(
            points=points,
            cells={"tetra": cells},
        ),
    )
   

# Outline

In [3]:
import gmsh
import os

# -------------------------------------------------------------------------
# Geometry dimensions [m]
# -------------------------------------------------------------------------

Lx = 500.0
Ly = 750.0
Lz = 125.0

# -------------------------------------------------------------------------
# Crevasse dimensions [m]
# -------------------------------------------------------------------------

lx = 5.0
ly = 10.0
lz = 10.0

# -------------------------------------------------------------------------
# Crevasse offsets
# -------------------------------------------------------------------------

x_offsets = [15.0, 25.0, 50.0, 100.0]

# -------------------------------------------------------------------------
# File numbering
# -------------------------------------------------------------------------

file_index = {
    15.0: 3,
    25.0: 4,
    50.0: 5,
    100.0: 6,
}

# -------------------------------------------------------------------------
# Outline mesh size
# -------------------------------------------------------------------------

h_min = 2.0
h_max = 20.0

# -------------------------------------------------------------------------
# Generate outline meshes
# -------------------------------------------------------------------------

for x_offset in x_offsets:

    gmsh.initialize()

    gmsh.model.add(
        f"glacier_terminus_{x_offset:.0f}_outline"
    )

    gmsh.option.setNumber(
        "Mesh.MeshSizeMin",
        h_min,
    )

    gmsh.option.setNumber(
        "Mesh.MeshSizeMax",
        h_max,
    )

    gmsh.option.setNumber(
        "Mesh.MeshSizeFactor",
        1.0,
    )

    # ---------------------------------------------------------------------
    # Glacier geometry
    # ---------------------------------------------------------------------

    glacier = gmsh.model.occ.addBox(
        0.0,
        0.0,
        0.0,
        Lx,
        Ly,
        Lz,
    )

    # ---------------------------------------------------------------------
    # Crevasse 1
    #
    # Enters from y = 0
    # ---------------------------------------------------------------------

    crevasse_1_x0 = (
        0.5 * Lx
        - x_offset
        - 0.5 * lx
    )

    crevasse_1_y0 = 0.0
    crevasse_1_z0 = Lz - lz

    crevasse_1 = gmsh.model.occ.addBox(
        crevasse_1_x0,
        crevasse_1_y0,
        crevasse_1_z0,
        lx,
        ly,
        lz,
    )

    # ---------------------------------------------------------------------
    # Crevasse 2
    #
    # Enters from y = Ly
    # ---------------------------------------------------------------------

    crevasse_2_x0 = (
        0.5 * Lx
        + x_offset
        - 0.5 * lx
    )

    crevasse_2_y0 = Ly - ly
    crevasse_2_z0 = Lz - lz

    crevasse_2 = gmsh.model.occ.addBox(
        crevasse_2_x0,
        crevasse_2_y0,
        crevasse_2_z0,
        lx,
        ly,
        lz,
    )

    # ---------------------------------------------------------------------
    # Subtract both crevasses
    # ---------------------------------------------------------------------

    domain, _ = gmsh.model.occ.cut(
        [(3, glacier)],
        [
            (3, crevasse_1),
            (3, crevasse_2),
        ],
        removeObject=True,
        removeTool=True,
    )

    gmsh.model.occ.synchronize()

    # ---------------------------------------------------------------------
    # Get all geometric curves
    # ---------------------------------------------------------------------

    curves = gmsh.model.getEntities(1)

    curve_tags = [
        tag
        for dim, tag in curves
    ]

    # ---------------------------------------------------------------------
    # Physical group for outline
    # ---------------------------------------------------------------------

    if curve_tags:

        gmsh.model.addPhysicalGroup(
            1,
            curve_tags,
            1,
        )

        gmsh.model.setPhysicalName(
            1,
            1,
            "OUTLINE",
        )

    # ---------------------------------------------------------------------
    # Generate ONLY 1D line mesh
    # ---------------------------------------------------------------------

    gmsh.model.mesh.generate(1)

    # ---------------------------------------------------------------------
    # Output filename
    #
    # 03/Lx500_2C_S15_outline.msh
    # 04/Lx500_2C_S25_outline.msh
    # 05/Lx500_2C_S50_outline.msh
    # 06/Lx500_2C_S100_outline.msh
    # ---------------------------------------------------------------------

    index = file_index[x_offset]

    os.makedirs(
        f"{index:02d}",
        exist_ok=True,
    )

    filename = (
        f"{index:02d}/"
        f"Lx{Lx:.0f}_"
        f"2C_"
        f"S{x_offset:.0f}_"
        f"outline.msh"
    )

    gmsh.write(filename)

    print(f"Generated: {filename}")

    gmsh.finalize()

Info    : Meshing 1D...                                                                                      
Info    : [  0%] Meshing curve 13 (Line)
Info    : [ 10%] Meshing curve 14 (Line)
Info    : [ 10%] Meshing curve 15 (Line)
Info    : [ 10%] Meshing curve 16 (Line)
Info    : [ 20%] Meshing curve 17 (Line)
Info    : [ 20%] Meshing curve 18 (Line)
Info    : [ 20%] Meshing curve 19 (Line)
Info    : [ 20%] Meshing curve 20 (Line)
Info    : [ 30%] Meshing curve 21 (Line)
Info    : [ 30%] Meshing curve 23 (Line)
Info    : [ 30%] Meshing curve 24 (Line)
Info    : [ 40%] Meshing curve 25 (Line)
Info    : [ 40%] Meshing curve 26 (Line)
Info    : [ 40%] Meshing curve 27 (Line)
Info    : [ 40%] Meshing curve 28 (Line)
Info    : [ 50%] Meshing curve 29 (Line)
Info    : [ 50%] Meshing curve 30 (Line)
Info    : [ 50%] Meshing curve 31 (Line)
Info    : [ 60%] Meshing curve 32 (Line)
Info    : [ 60%] Meshing curve 33 (Line)
Info    : [ 60%] Meshing curve 34 (Line)
Info    : [ 60%] Meshing curv

Info    : [ 20%] Meshing curve 19 (Line)
Info    : [ 20%] Meshing curve 20 (Line)
Info    : [ 30%] Meshing curve 21 (Line)
Info    : [ 30%] Meshing curve 23 (Line)
Info    : [ 30%] Meshing curve 24 (Line)
Info    : [ 40%] Meshing curve 25 (Line)
Info    : [ 40%] Meshing curve 26 (Line)
Info    : [ 40%] Meshing curve 27 (Line)
Info    : [ 40%] Meshing curve 28 (Line)
Info    : [ 50%] Meshing curve 29 (Line)
Info    : [ 50%] Meshing curve 30 (Line)
Info    : [ 50%] Meshing curve 31 (Line)
Info    : [ 60%] Meshing curve 32 (Line)
Info    : [ 60%] Meshing curve 33 (Line)
Info    : [ 60%] Meshing curve 34 (Line)
Info    : [ 60%] Meshing curve 35 (Line)
Info    : [ 70%] Meshing curve 36 (Line)
Info    : [ 70%] Meshing curve 37 (Line)
Info    : [ 70%] Meshing curve 38 (Line)
Info    : [ 70%] Meshing curve 39 (Line)
Info    : [ 80%] Meshing curve 40 (Line)
Info    : [ 80%] Meshing curve 41 (Line)
Info    : [ 80%] Meshing curve 42 (Line)
Info    : [ 90%] Meshing curve 43 (Line)
Info    : [ 90%]

# Outline - XDMF

In [4]:
import meshio

mesh_files = ["03/Lx500_2C_S15_outline.msh", "04/Lx500_2C_S25_outline.msh", "05/Lx500_2C_S50_outline.msh", "06/Lx500_2C_S100_outline.msh"]

for mesh_file in mesh_files:
    mesh = meshio.read(mesh_file)
    cells = mesh.get_cells_type("line")
    points = mesh.points

    meshio.write(
        mesh_file.replace(".msh", ".xdmf"),
        meshio.Mesh(
            points=points,
            cells={"line": cells},
        ),
    )
   